In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt


from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# 1. Thay "HF_TOKEN" bằng đúng cái tên (Label) bạn đã đặt trong Add-ons
secret_label = "HF_TOKEN" 
secret_value = UserSecretsClient().get_secret(secret_label)

# 2. Dùng giá trị secret lấy được để đăng nhập tự động vào Hugging Face
login(token=secret_value, add_to_git_credential=False)

In [2]:
# ============================================================
# CELL OCR-1: INSTALL + IMPORT
# ============================================================
!pip install python-doctr[torch] rank_bm25 -q

import os, json, re, gc
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from datetime import datetime
from datasets import load_dataset

from doctr.models import ocr_predictor
from rank_bm25 import BM25Okapi

import os, glob, zipfile

INPUT_ROOT       = "/kaggle/input"
WORK_ARTIFACT_DIR = "/kaggle/working/docvqa_colqwen2_artifacts"

zip_candidates = [
    p for p in glob.glob(f"{INPUT_ROOT}/**/*.zip", recursive=True)
    if "docvqa_colqwen2_artifacts" in os.path.basename(p)
]

if not os.path.exists(f"{WORK_ARTIFACT_DIR}/manifest.json"):
    assert len(zip_candidates) > 0, (
        "❌ Không tìm thấy file zip artifact trong /kaggle/input — "
        "kiểm tra lại dataset 'Re-2-MMRAG' đã được add vào Notebook chưa."
    )
    zip_path = zip_candidates[0]
    print(f"📦 Giải nén artifact từ: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("/kaggle/working")
    print("✅ Đã giải nén vào", WORK_ARTIFACT_DIR)
else:
    print("✅ Artifact đã sẵn có trong /kaggle/working")

ARTIFACT_DIR = WORK_ARTIFACT_DIR
assert os.path.exists(f"{ARTIFACT_DIR}/manifest.json"), "manifest.json vẫn không thấy!"
print("Artifact dir:", ARTIFACT_DIR)
print("Files       :", sorted(os.listdir(ARTIFACT_DIR)))


# ============================================================
# CELL OCR-2: CONFIG
# ============================================================
DATASET_NAME = "vidore/docvqa_train"

OCR_SHARD_DIR  = f"{ARTIFACT_DIR}/ocr_text_shards_doctr"   # đổi tên dir để không đè lên shard EasyOCR cũ
OCR_SHARD_SIZE = 200
RETRIEVAL_DIR  = f"{ARTIFACT_DIR}/retrieval_results"

os.makedirs(OCR_SHARD_DIR, exist_ok=True)
os.makedirs(RETRIEVAL_DIR, exist_ok=True)

# Giữ nguyên tên file E1 — để các notebook E2/E4 phía sau không cần sửa gì
E1_RESULTS_PATH = f"{RETRIEVAL_DIR}/e1_bm25_topk_results.json"
E1_METRICS_PATH = f"{RETRIEVAL_DIR}/e1_bm25_metrics.json"

TOP_K          = 10
OCR_MAX_SIDE   = 1600
OCR_BATCH_SIZE = 16     # docTR forward theo batch — tận dụng GPU tốt hơn EasyOCR xử lý từng ảnh


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.9/288.9 kB 19.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, 

In [3]:
# ============================================================
# CELL OCR-3: LOAD METADATA ĐÃ CÓ TỪ STEP 1 + 2
# ============================================================
with open(f"{ARTIFACT_DIR}/corpus_ids.json", "r", encoding="utf-8") as f:
    corpus_ids = json.load(f)

with open(f"{ARTIFACT_DIR}/corpus_id_to_idx.json", "r", encoding="utf-8") as f:
    corpus_id_to_idx = json.load(f)

with open(f"{ARTIFACT_DIR}/qa_pairs.json", "r", encoding="utf-8") as f:
    qa_pairs = json.load(f)

with open(f"{ARTIFACT_DIR}/manifest.json", "r", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"Corpus images : {len(corpus_ids):,}")
print(f"QA pairs      : {len(qa_pairs):,}")

print("\nĐang tải lại dataset gốc để lấy ảnh cho OCR...")
full_ds = load_dataset(DATASET_NAME, split="train")
print(f"✅ full_ds: {len(full_ds):,} records — khớp với corpus_id_to_idx đã lưu")


# ============================================================
# CELL OCR-4 (DOCTR): KHỞI TẠO PREDICTOR + HELPER
# ============================================================
print("\nĐang khởi tạo docTR predictor...")
doctr_model = ocr_predictor(
    det_arch='db_resnet50',     # detection — phù hợp cho ảnh document/scan
    reco_arch='parseq',         # recognition — chính xác hơn CRNN mặc định
    pretrained=True
)
if torch.cuda.is_available():
    doctr_model = doctr_model.cuda()
print(f"✅ docTR predictor sẵn sàng (gpu={torch.cuda.is_available()})")


def safe_load_image_np(img, max_side=OCR_MAX_SIDE):
    """
    Chuyển PIL image → numpy array RGB cho docTR.
    Resize nếu quá lớn để tăng tốc OCR, fallback ảnh trắng nếu lỗi.
    """
    try:
        if img is None:
            raise ValueError("None image")
        img = img.convert("RGB")
        if max(img.size) > max_side:
            img.thumbnail((max_side, max_side), Image.LANCZOS)
        return np.array(img)
    except Exception:
        return np.array(Image.new("RGB", (224, 224), color=(255, 255, 255)))


def _page_to_text(page):
    """Ghép text theo thứ tự block → line → word, giữ ngữ cảnh câu tốt hơn việc join box rời rạc."""
    lines = []
    for block in page.blocks:
        for line in block.lines:
            lines.append(" ".join(w.value for w in line.words))
    return " ".join(lines).strip()


def ocr_extract_batch(images_np_list):
    """
    Chạy docTR trên 1 batch ảnh cùng lúc — nhanh hơn nhiều so với
    xử lý từng ảnh một (đặc biệt trên GPU).
    """
    try:
        result = doctr_model(images_np_list)
        return [_page_to_text(page) for page in result.pages]
    except Exception as e:
        print("⚠️ OCR batch error:", repr(e))
        return ["" for _ in images_np_list]


# ============================================================
# CELL OCR-5: SHARD UTILS (cùng pattern với corpus embedding)
# ============================================================
def ocr_shard_exists(shard_idx):
    return os.path.exists(f"{OCR_SHARD_DIR}/shard_{shard_idx:05d}.json")


def save_ocr_shard(shard_idx, shard_ids, shard_texts):
    path = f"{OCR_SHARD_DIR}/shard_{shard_idx:05d}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump({"ids": shard_ids, "texts": shard_texts}, f, ensure_ascii=False)
    return path


def load_ocr_shard(shard_idx):
    path = f"{OCR_SHARD_DIR}/shard_{shard_idx:05d}.json"
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data["ids"], data["texts"]


# ============================================================
# CELL OCR-6 (DOCTR): RUN OCR TRÊN TOÀN BỘ CORPUS — resume-able theo shard, batched
# ============================================================
def run_ocr_corpus(corpus_ids, corpus_id_to_idx, dataset,
                    shard_size=OCR_SHARD_SIZE, batch_size=OCR_BATCH_SIZE, resume=True):

    num_docs   = len(corpus_ids)
    num_shards = int(np.ceil(num_docs / shard_size))

    print("\n========== START OCR EXTRACTION (docTR) ==========")
    print(f"Corpus size : {num_docs:,}")
    print(f"Num shards  : {num_shards}")
    print(f"Shard size  : {shard_size}")
    print(f"Batch size  : {batch_size}")
    print(f"OCR max side: {OCR_MAX_SIDE}px")
    print("====================================================")

    for shard_idx in range(num_shards):
        if resume and ocr_shard_exists(shard_idx):
            continue

        s = shard_idx * shard_size
        e = min((shard_idx + 1) * shard_size, num_docs)
        cur_ids = corpus_ids[s:e]

        shard_texts = []
        for b_start in tqdm(range(0, len(cur_ids), batch_size),
                             desc=f"OCR shard {shard_idx:03d}/{num_shards}", leave=False):
            b_end = min(b_start + batch_size, len(cur_ids))
            batch_ids = cur_ids[b_start:b_end]

            batch_imgs = [
                safe_load_image_np(dataset[corpus_id_to_idx[doc_id]]["image"])
                for doc_id in batch_ids
            ]
            batch_texts = ocr_extract_batch(batch_imgs)
            shard_texts.extend(batch_texts)

        save_ocr_shard(shard_idx, cur_ids, shard_texts)

        if shard_idx % 5 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    print("\n🎉 OCR extraction hoàn tất!")


# Chạy OCR — resume=True để tiếp tục nếu bị crash giữa chừng
run_ocr_corpus(corpus_ids, corpus_id_to_idx, full_ds,
               shard_size=OCR_SHARD_SIZE, batch_size=OCR_BATCH_SIZE, resume=True)


# ============================================================
# CELL OCR-7: LOAD TOÀN BỘ OCR TEXT + TOKENIZE CHO BM25
# ============================================================
def load_all_ocr_texts(num_shards):
    all_ids, all_texts = [], []
    for shard_idx in tqdm(range(num_shards), desc="Loading OCR shards"):
        ids, texts = load_ocr_shard(shard_idx)
        all_ids.extend(ids)
        all_texts.extend(texts)
    return all_ids, all_texts


def simple_tokenize(text):
    """Lowercase + tách từ alphanumeric — đủ dùng cho BM25 tiếng Anh."""
    return re.findall(r"[a-z0-9]+", text.lower())


num_ocr_shards = int(np.ceil(len(corpus_ids) / OCR_SHARD_SIZE))
ocr_ids, ocr_texts = load_all_ocr_texts(num_ocr_shards)

assert ocr_ids == corpus_ids, "❌ Thứ tự OCR ids không khớp corpus_ids — kiểm tra lại sharding"

print(f"\n✅ Loaded OCR text cho {len(ocr_texts):,} images")
print("Ví dụ OCR text[0][:200]:", repr(ocr_texts[0][:200]))

tokenized_corpus = [simple_tokenize(t) for t in tqdm(ocr_texts, desc="Tokenizing corpus")]

empty_count = sum(1 for toks in tokenized_corpus if len(toks) == 0)
print(f"⚠️ Số ảnh OCR rỗng (0 token): {empty_count} / {len(tokenized_corpus)} "
      f"({empty_count/len(tokenized_corpus)*100:.1f}%)")


# ============================================================
# CELL OCR-8: BUILD BM25 INDEX + LƯU OCR CORPUS ĐẦY ĐỦ
# ============================================================
print("\nĐang build BM25 index...")
bm25 = BM25Okapi(tokenized_corpus)
print("✅ BM25 index sẵn sàng")

# Giữ nguyên tên file + key — để E2/E4 load lại được không cần sửa code
with open(f"{ARTIFACT_DIR}/ocr_corpus_full.json", "w", encoding="utf-8") as f:
    json.dump({"corpus_ids": ocr_ids, "ocr_texts": ocr_texts}, f, ensure_ascii=False)
print(f"💾 Đã lưu: {ARTIFACT_DIR}/ocr_corpus_full.json")


# ============================================================
# CELL OCR-9: METRICS (giữ nguyên logic từ E0)
# ============================================================
def compute_retrieval_metrics(results, k_values=[1, 5, 10]):
    metrics = {}
    n = len(results)

    for k in k_values:
        hit = sum(
            1 for item in results
            if item["ground_truth_image_filename"] in
               [x["image_id"] for x in item["topk"][:k]]
        )
        metrics[f"Recall@{k}"] = hit / n

    rr_list = []
    for item in results:
        gt = item["ground_truth_image_filename"]
        for rank, x in enumerate(item["topk"][:10], 1):
            if x["image_id"] == gt:
                rr_list.append(1.0 / rank); break
        else:
            rr_list.append(0.0)
    metrics["MRR@10"] = float(np.mean(rr_list))

    ndcg_list = []
    for item in results:
        gt = item["ground_truth_image_filename"]
        for rank, x in enumerate(item["topk"][:10], 1):
            if x["image_id"] == gt:
                ndcg_list.append(1.0 / np.log2(rank + 1)); break
        else:
            ndcg_list.append(0.0)
    metrics["NDCG@10"] = float(np.mean(ndcg_list))

    return metrics


# ============================================================
# CELL OCR-10: RUN E1 — BM25 RETRIEVAL CHO 2,000 VAL QUERIES (trên OCR docTR)
# ============================================================
def run_e1_bm25_retrieval(qa_pairs, bm25, corpus_ids, top_k=10, resume=True):

    if resume and os.path.exists(E1_RESULTS_PATH) and os.path.exists(E1_METRICS_PATH):
        print("✅ E1 results đã tồn tại, load lại.")
        with open(E1_RESULTS_PATH, encoding="utf-8") as f: results = json.load(f)
        with open(E1_METRICS_PATH, encoding="utf-8")  as f: metrics_info = json.load(f)
        print("Metrics:", metrics_info["metrics"])
        return results, metrics_info

    results = []
    for i, item in enumerate(tqdm(qa_pairs, desc="E1 BM25 Retrieval (docTR)")):
        query_tokens = simple_tokenize(item["query"])
        scores = bm25.get_scores(query_tokens)

        top_idx = np.argsort(scores)[::-1][:top_k]
        topk = [
            {"rank": r + 1, "image_id": corpus_ids[idx], "score": float(scores[idx])}
            for r, idx in enumerate(top_idx)
        ]

        results.append({
            "query_index": i,
            "query": item["query"],
            "ground_truth_image_filename": item["image_filename"],
            "answer": item["answer"],
            "topk": topk,
        })

    metrics = compute_retrieval_metrics(results, k_values=[1, 5, 10])
    metrics_info = {
        "experiment": "E1_DocTR_BM25_Sparse_Retrieval",   # ghi rõ engine OCR để phân biệt với bản EasyOCR
        "ocr_engine": "doctr (det_arch=db_resnet50, reco_arch=parseq)",
        "num_queries": len(results),
        "top_k": top_k,
        "created_at": datetime.now().isoformat(),
        "metrics": metrics,
    }

    with open(E1_RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    with open(E1_METRICS_PATH, "w", encoding="utf-8") as f:
        json.dump(metrics_info, f, ensure_ascii=False, indent=2)

    print("\n✅ E1 (docTR) BM25 Retrieval finished!")
    print("Results:", E1_RESULTS_PATH)
    print("Metrics:", E1_METRICS_PATH)
    for k, v in metrics.items():
        print(f"   {k}: {v:.4f}")

    return results, metrics_info


e1_results, e1_metrics = run_e1_bm25_retrieval(
    qa_pairs=qa_pairs,
    bm25=bm25,
    corpus_ids=corpus_ids,
    top_k=TOP_K,
    resume=False,
)


# ============================================================
# CELL OCR-11: CHECK KẾT QUẢ MẪU
# ============================================================
print("\n========== CHECK E1 (docTR) RESULTS ==========")
example = e1_results[0]

print("Query:", example["query"])
print("GT   :", example["ground_truth_image_filename"])
print("TopK :")
for item in example["topk"][:5]:
    marker = "✅" if item["image_id"] == example["ground_truth_image_filename"] else "  "
    print(f"{item['rank']:>2}. {marker} {item['image_id']} | score={item['score']:.4f}")
print("=================================================")

Corpus images : 10,189
QA pairs      : 2,000

Đang tải lại dataset gốc để lấy ảnh cho OCR...


README.md:   0%|          | 0.00/516 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/57 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/57 [00:00<?, ?it/s]

data/train-00000-of-00057.parquet:   0%|          | 0.00/26.8M [00:00<?, ?B/s]

...


✅ Loaded OCR text cho 10,189 images
Ví dụ OCR text[0][:200]: 'Cafidatial RJRT PR APPROVAL DATE: ilx/13 BUBJECT: Ru glas PROPOBED RELEASE DATE: fo respmse FOR RELBASE TO: CONTACT: P.CabreR ROUTE TOI Name Initiala Data Paggy Carter Au ilries Maura Payne int David '


Tokenizing corpus:   0%|          | 0/10189 [00:00<?, ?it/s]

⚠️ Số ảnh OCR rỗng (0 token): 0 / 10189 (0.0%)

Đang build BM25 index...
✅ BM25 index sẵn sàng
💾 Đã lưu: /kaggle/working/docvqa_colqwen2_artifacts/ocr_corpus_full.json


E1 BM25 Retrieval (docTR):   0%|          | 0/2000 [00:00<?, ?it/s]


✅ E1 (docTR) BM25 Retrieval finished!
Results: /kaggle/working/docvqa_colqwen2_artifacts/retrieval_results/e1_bm25_topk_results.json
Metrics: /kaggle/working/docvqa_colqwen2_artifacts/retrieval_results/e1_bm25_metrics.json
   Recall@1: 0.1555
   Recall@5: 0.2375
   Recall@10: 0.2705
   MRR@10: 0.1907
   NDCG@10: 0.2097

========== CHECK E1 (docTR) RESULTS ==========
Query: What is the Submitting Office No.?
Provide a short and direct response.
GT   : 4b58b09cf9f18f9f32b056949d50847e3cbf98e60042b0606617adaf69bb8a00
TopK :
 1.    bbb44b654fa559fc8beb3a7425d3184e9ee1a18fe5737dfc213e70ae2995b5c1 | score=26.6793
 2.    0ea2bb8b2faff63d55c121aab07fde921a8f6e97c61caa687261111e4e01747d | score=23.8918
 3.    909f9df1cac97530c056fa33c32531f586025dba335771caa0e3c4944adc7464 | score=23.1883
 4.    bbc0f49625d0d7b50e8a184fb89dddb790028e320de59332a7af6a6df598193a | score=23.1384
 5.    6aa57b5dfbc639a6e57b7faac7442136daea556ab45e6d208df9ca8cc298ab83 | score=22.8829


In [4]:
# ============================================================
# CELL OCR-12: ZIP LẠI ARTIFACT (bao gồm OCR docTR + E1 mới)
# ============================================================
import shutil

zip_base = "/kaggle/working/docvqa_colqwen2_artifacts"
zip_path = f"{zip_base}.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir="/kaggle/working",
    base_dir="docvqa_colqwen2_artifacts"
)

print("✅ Đã zip artifact (gồm OCR docTR + E1 mới):")
print(zip_path)
print("File size:", round(os.path.getsize(zip_path) / (1024*1024), 2), "MB")

# Check nhanh các file quan trọng có trong zip chưa
with zipfile.ZipFile(zip_path, "r") as zf:
    names = zf.namelist()
    for fname in ["ocr_corpus_full.json", "e1_bm25_topk_results.json", "e1_bm25_metrics.json"]:
        found = any(n.endswith(fname) for n in names)
        print(f"{'✅' if found else '❌'} {fname}")

print("\n⚠️  Nhớ: vào tab Output → Download/Save version, rồi upload file zip này")
print("    làm 1 Kaggle Dataset mới (hoặc update dataset cũ), để E2/E4 dùng tiếp.")

✅ Đã zip artifact (gồm OCR docTR + E1 mới):
/kaggle/working/docvqa_colqwen2_artifacts.zip
File size: 1496.87 MB
✅ ocr_corpus_full.json
✅ e1_bm25_topk_results.json
✅ e1_bm25_metrics.json

⚠️  Nhớ: vào tab Output → Download/Save version, rồi upload file zip này
    làm 1 Kaggle Dataset mới (hoặc update dataset cũ), để E2/E4 dùng tiếp.
